# Week 13 demo: where does the human go?
Pre-run. No live prompting. **The AI extraction in `ai_extraction.tsv` is a stand-in, not a real model run:** errors of realistic kinds were planted at known rows so we have a known answer. All data are synthetic.

Task the "AI" did: read 60 messy sample records (`raw_samples.csv`) and return a clean table. Then a downstream step computes mean glucose by sex.

In [1]:
import pandas as pd, numpy as np, datetime, random, os
from hitl_log import log_event, read_log, sha
REF = datetime.date(2026, 11, 16)
raw   = pd.read_csv("raw_samples.csv")
ai    = pd.read_csv("ai_extraction.tsv", sep="\t")
truth = pd.read_csv("truth.tsv", sep="\t")
cols = ["dob","sex","site","glucose_mgdl"]
def wrong_rows(df):
    m = df.merge(truth, on="sample_id", suffixes=("","_t"))
    bad = np.zeros(len(m), bool)
    for c in cols:
        a, t = m[c], m[c+"_t"]
        bad |= ~((a == t) | (a.isna() & t.isna()))
    return set(m.sample_id[bad])
planted = pd.read_csv("planted_errors.tsv", sep="\t")
bad_all = wrong_rows(ai)
print("rows:", len(ai), "| rows with at least one wrong field:", len(bad_all))
print("matches the planted list:", bad_all == set(planted.sample_id))
planted.error.value_counts().to_frame("rows")

rows: 60 | rows with at least one wrong field: 13
matches the planted list: True


,rows
error,
date_swap,4
unit_not_converted,3
two_digit_year,2
sex_guess,2
impute_na,1
site_error,1


## 0. No checkpoint: what the downstream step sees

In [2]:
def by_sex(df):
    return df.groupby("sex").glucose_mgdl.agg(n="count", mean="mean").round(1)
pd.concat({"AI output (no checkpoint)": by_sex(ai), "truth": by_sex(truth)}, axis=1)

AI output (no checkpoint)        truth       
                            n   mean     n   mean
sex                                              
F                          18  148.4    17  151.6
M                          24  157.3    22  171.2
U                          15  154.2    17  167.7

## 1. Automatic validators (cheap, run on every row)
Rules look only at the AI's output. No human reads anything.

In [3]:
def validate(df):
    flags = []
    for _, r in df.iterrows():
        d = datetime.date.fromisoformat(r.dob)
        if d > REF:                                   flags.append((r.sample_id, "dob in the future"))
        if r.sex not in ("M","F","U"):                flags.append((r.sample_id, "sex not in M/F/U"))
        if r.site not in ("A","B","C"):               flags.append((r.sample_id, "unknown site"))
        if pd.notna(r.glucose_mgdl) and not (40 <= r.glucose_mgdl <= 600):
                                                      flags.append((r.sample_id, "glucose outside 40-600 mg/dL"))
    return pd.DataFrame(flags, columns=["sample_id","rule"])
f1 = validate(ai)
s1 = set(f1.sample_id)
print("flagged:", len(s1), "| real errors caught:", len(s1 & bad_all), "| false alarms:", len(s1 - bad_all))
print("errors still in the data:", len(bad_all - s1))
f1.merge(planted, on="sample_id").groupby(["rule","error"]).size().to_frame("rows")

flagged: 5 | real errors caught: 5 | false alarms: 0
errors still in the data: 8


,,rows
rule,error,
dob in the future,two_digit_year,2
glucose outside 40-600 mg/dL,unit_not_converted,3


In [4]:
# What the validators cannot see: still wrong, perfectly valid-looking
missed = planted[~planted.sample_id.isin(s1)].merge(ai, on="sample_id").merge(truth, on="sample_id", suffixes=("_ai","_true"))
missed[["sample_id","error","dob_ai","dob_true","sex_ai","sex_true","site_ai","site_true","glucose_mgdl_ai","glucose_mgdl_true","ai_confidence"]]

,sample_id,error,dob_ai,dob_true,sex_ai,sex_true,site_ai,site_true,glucose_mgdl_ai,glucose_mgdl_true,ai_confidence
0,S0033,date_swap,1951-06-12,1951-12-06,F,F,A,A,160.3,160.3,0.95
1,S0051,date_swap,1988-11-10,1988-10-11,U,U,B,B,108.1,108.1,0.92
2,S0042,date_swap,1956-02-01,1956-01-02,F,F,A,A,248.5,248.5,0.95
3,S0012,date_swap,1988-02-07,1988-07-02,U,U,C,C,186.1,186.1,0.92
4,S0057,sex_guess,1951-06-29,1951-06-29,F,U,B,B,93.7,93.7,0.80
5,S0018,sex_guess,1977-06-27,1977-06-27,M,U,B,B,140.5,140.5,0.85
6,S0027,impute_na,1974-01-08,1974-01-08,M,M,B,B,100.0,NaN,0.87
7,S0054,site_error,1953-11-22,1953-11-22,M,M,C,A,78.0,78.0,0.90


## 2. Confidence routing: send low-confidence rows to a human

In [5]:
THRESH = 0.80
s2 = set(ai.sample_id[ai.ai_confidence < THRESH]) - s1
print("extra rows routed:", len(s2), "| real errors among them:", len(s2 & bad_all), "| false alarms:", len(s2 - bad_all))
left = bad_all - s1 - s2
print("errors still in the data:", len(left))
print("confidence of the errors left:", sorted(ai.set_index("sample_id").loc[sorted(left), "ai_confidence"].tolist()))
print("mean confidence, correct rows:", round(ai[~ai.sample_id.isin(bad_all)].ai_confidence.mean(),3),
      "| wrong rows:", round(ai[ai.sample_id.isin(bad_all)].ai_confidence.mean(),3))

extra rows routed: 4 | real errors among them: 0 | false alarms: 4
errors still in the data: 8
confidence of the errors left: [0.8, 0.85, 0.87, 0.9, 0.92, 0.92, 0.95, 0.95]
mean confidence, correct rows: 0.92 | wrong rows: 0.806


## 3. Random audit sample of what passed every gate

In [6]:
passed = sorted(set(ai.sample_id) - s1 - s2)
k = 6                                  # 10% of 60
rng = random.Random(0)
hits = [len(set(rng.sample(passed, k)) & left) for _ in range(5000)]
print(f"{len(passed)} rows passed; {len(left)} are wrong ({len(left)/len(passed):.0%}). Audit sample of {k}.")
print("chance the sample finds at least one error:", round(np.mean([h>0 for h in hits]),2))
print("mean errors found per sample:", round(np.mean(hits),2), "of", len(left))

51 rows passed; 8 are wrong (16%). Audit sample of 6.
chance the sample finds at least one error: 0.66
mean errors found per sample: 0.93 of 8


## 4. Compare designs: rows a human reads vs. errors left
Perfect reviewer assumed in the first table. Second table: the reviewer misses 20% of the errors they look at.

In [7]:
def design(name, read, fixed_by_auto=0):
    caught = len(read & bad_all)
    return dict(design=name, rows_read=len(read), errors_caught=caught, errors_left=len(bad_all)-caught)
all_rows = set(ai.sample_id)
d = [design("1. No checkpoint", set()),
     design("2. Validators, human fixes flagged rows", s1),
     design("3. + low-confidence rows", s1|s2),
     design("4. + audit sample (expected)", s1|s2),   # sample adds expected hits, see below
     design("5. Read everything", all_rows)]
d[3]["errors_caught"] += round(np.mean(hits),1); d[3]["errors_left"] = round(d[3]["errors_left"]-np.mean(hits),1); d[3]["rows_read"] += k
t = pd.DataFrame(d)
t["errors_left_if_20%_missed"] = (t.errors_left + 0.2*t.errors_caught).round(1)
t

,design,rows_read,errors_caught,errors_left,errors_left_if_20%_missed
0,1. No checkpoint,0,0.0,13.0,13.0
1,"2. Validators, human fixes flagged rows",5,5.0,8.0,9.0
2,3. + low-confidence rows,9,5.0,8.0,9.0
3,4. + audit sample (expected),15,5.9,7.1,8.3
4,5. Read everything,60,13.0,0.0,2.6


## 5. Same downstream step after design 3

In [8]:
fixed = ai.set_index("sample_id").copy()
for sid in (s1|s2):
    fixed.loc[sid, cols] = truth.set_index("sample_id").loc[sid, cols].values   # reviewer corrects from the source
fixed = fixed.reset_index()
pd.concat({"no checkpoint": by_sex(ai), "after design 3": by_sex(fixed), "truth": by_sex(truth)}, axis=1)

no checkpoint        after design 3        truth       
                n   mean              n   mean     n   mean
sex                                                        
F              18  148.4             18  148.4    17  151.6
M              24  157.3             24  167.0    22  171.2
U              15  154.2             15  174.5    17  167.7

## 6. The audit log: what to record

In [9]:
LOG = "demo_audit_log.jsonl"
if os.path.exists(LOG): os.remove(LOG)
log_event(LOG, "extract", "ai:stand-in", input_sha=sha("raw_samples.csv"), output_sha=sha("ai_extraction.tsv"),
          prompt="(stand-in; in real use, save the prompt text or its hash)", n_rows=len(ai))
log_event(LOG, "validate", "code:validate()", rules=4, flagged=len(s1))
log_event(LOG, "route", "code:confidence<0.80", routed=len(s2))
for sid in sorted(s1|s2):
    t_row, a_row = truth.set_index("sample_id").loc[sid], ai.set_index("sample_id").loc[sid]
    changed = [c for c in cols if not (a_row[c]==t_row[c] or (pd.isna(a_row[c]) and pd.isna(t_row[c])))]
    log_event(LOG, "review", "human:MB", sample_id=sid, decision="fix" if changed else "accept",
              changed_fields=changed, reason="checked against raw row" if changed else "matches source")
log = pd.DataFrame(read_log(LOG))
log.head(8)

,ts,step,actor,input_sha,output_sha,prompt,n_rows,rules,flagged,routed,sample_id,decision,changed_fields,reason
0,2026-11-16T10:38:02,extract,ai:stand-in,1a20c73f7e34,c3c7204b3d72,"(stand-in; in real use, save the prompt text o...",60.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-11-16T10:38:02,validate,code:validate(),NaN,NaN,NaN,NaN,4.0,5.0,NaN,NaN,NaN,NaN,NaN
2,2026-11-16T10:38:02,route,code:confidence<0.80,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN
3,2026-11-16T10:38:02,review,human:MB,NaN,NaN,NaN,NaN,NaN,NaN,NaN,S0002,accept,[],matches source
4,2026-11-16T10:38:02,review,human:MB,NaN,NaN,NaN,NaN,NaN,NaN,NaN,S0006,accept,[],matches source
5,2026-11-16T10:38:02,review,human:MB,NaN,NaN,NaN,NaN,NaN,NaN,NaN,S0016,accept,[],matches source
6,2026-11-16T10:38:02,review,human:MB,NaN,NaN,NaN,NaN,NaN,NaN,NaN,S0040,accept,[],matches source
7,2026-11-16T10:38:02,review,human:MB,NaN,NaN,NaN,NaN,NaN,NaN,NaN,S0052,fix,[dob],checked against raw row


In [10]:
# Two questions an auditor (or you, in 3 months) should be able to answer from the log alone
print("Which values did a human change?")
print(log[(log.step=="review") & (log.decision=="fix")][["sample_id","changed_fields","reason"]].to_string(index=False))
print()
print("Reviews that accepted the AI value unchanged:", int(((log.step=="review") & (log.decision=="accept")).sum()))

Which values did a human change?
sample_id changed_fields                  reason
    S0052          [dob] checked against raw row
    S0053 [glucose_mgdl] checked against raw row
    S0055 [glucose_mgdl] checked against raw row
    S0056          [dob] checked against raw row
    S0060 [glucose_mgdl] checked against raw row

Reviews that accepted the AI value unchanged: 4
